# Parameter Sweep Studies: Gas Swelling in Nuclear Fuel

This notebook demonstrates how to perform **comprehensive parameter sweep studies** to investigate how different parameters affect fission gas bubble evolution and swelling in nuclear fuel materials.

## Learning Objectives

By the end of this tutorial, you will:
- Understand which parameters most strongly affect swelling
- Learn systematic approaches to parameter exploration
- Create parameter sensitivity plots
- Identify critical parameter thresholds and nonlinear behaviors
- Generate publication-quality parametric study visualizations

## Why Parameter Sweeps?

Parameter sweeps are essential for:
1. **Sensitivity analysis**: Identify which parameters most affect predictions
2. **Model validation**: Compare with experimental data across conditions
3. **Design optimization**: Find optimal operating conditions
4. **Uncertainty quantification**: Understand prediction confidence intervals
5. **Physical insight**: Reveal underlying mechanisms and couplings

## Parameters We'll Explore

- **Temperature**: Affects diffusion rates and bubble stability
- **Fission rate**: Controls gas production rate
- **Dislocation density**: Influences defect sink strength
- **Surface energy**: Affects bubble size and nucleation
- **Nucleation factors**: Control bubble number density
- **Gas production rate**: Fission gas yield per fission

---

In [ ]:
# Import required libraries
import numpy as np
import matplotlib.pyplot as plt
from gas_swelling.models.modelrk23 import GasSwellingModel
from gas_swelling.params.parameters import create_default_parameters
import warnings
warnings.filterwarnings('ignore')

# Configure plotting for better visuals
plt.rcParams['figure.figsize'] = (14, 6)
plt.rcParams['font.size'] = 10
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.3

print("Libraries imported successfully!")
print("\nReady to run parameter sweep studies.")

## 1. Setup: Define Helper Functions

First, let's create reusable functions for running parameter sweeps. This will make our code cleaner and more maintainable.

In [ ]:
def run_single_simulation(params, sim_time_days=100, n_points=100):
    """
    Run a single simulation with given parameters.
    
    Parameters:
    -----------
    params : dict
        Model parameters
    sim_time_days : float
        Simulation time in days
    n_points : int
        Number of time points for output
    
    Returns:
    --------
    dict : Results including time series and final values
    """
    # Create model
    model = GasSwellingModel(params)
    
    # Set up time
    sim_time_sec = sim_time_days * 24 * 3600
    t_eval = np.linspace(0, sim_time_sec, n_points)
    
    # Solve
    result = model.solve(t_span=(0, sim_time_sec), t_eval=t_eval)
    
    # Calculate swelling
    Vb = (4/3) * np.pi * result['Rcb']**3 * result['Ccb']
    Vf = (4/3) * np.pi * result['Rcf']**3 * result['Ccf']
    swelling = (Vb + Vf) * 100  # Convert to percent
    
    # Add swelling to result
    result['swelling'] = swelling
    result['time_days'] = result['time'] / (24 * 3600)
    
    return result


def run_parameter_sweep(param_name, param_values, base_params=None, sim_time_days=100):
    """
    Run a parameter sweep across a range of values.
    
    Parameters:
    -----------
    param_name : str
        Name of parameter to vary
    param_values : array
        Values to test
    base_params : dict
        Base parameters (default if None)
    sim_time_days : float
        Simulation time in days
    
    Returns:
    --------
    list : List of result dictionaries
    """
    if base_params is None:
        base_params = create_default_parameters()
    
    results = []
    
    print(f"Running {param_name} sweep...")
    for i, value in enumerate(param_values):
        # Create parameters with this value
        params = base_params.copy()
        params[param_name] = value
        
        # Run simulation
        result = run_single_simulation(params, sim_time_days)
        result[param_name] = value
        results.append(result)
        
        print(f"  [{i+1}/{len(param_values)}] {param_name}={value:.2e}: "
              f"swelling={result['swelling'][-1]:.4f}%")
    
    print("Done!\n")
    return results


def plot_sweep_results(results, param_name, param_label=None):
    """
    Create comprehensive plots from parameter sweep results.
    
    Parameters:
    -----------
    results : list
        List of result dictionaries from run_parameter_sweep
    param_name : str
        Name of parameter that was varied
    param_label : str
        Label for plots (default: param_name)
    """
    if param_label is None:
        param_label = param_name
    
    # Extract final values
    param_values = [r[param_name] for r in results]
    final_swelling = [r['swelling'][-1] for r in results]
    final_Rcb = [r['Rcb'][-1] * 1e9 for r in results]
    final_Rcf = [r['Rcf'][-1] * 1e9 for r in results]
    final_Ccb = [r['Ccb'][-1] for r in results]
    final_Ccf = [r['Ccf'][-1] for r in results]
    
    # Create figure with subplots
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    fig.suptitle(f'Effect of {param_label} on Gas Swelling', 
                 fontsize=14, fontweight='bold')
    
    # Plot 1: Final swelling vs parameter
    axes[0, 0].plot(param_values, final_swelling, 'o-', 
                   linewidth=2, markersize=8, color='red')
    axes[0, 0].set_xlabel(param_label)
    axes[0, 0].set_ylabel('Final Swelling (%)')
    axes[0, 0].set_title('Swelling Dependence')
    axes[0, 0].grid(True, alpha=0.3)
    
    # Plot 2: Bubble radius vs parameter
    axes[0, 1].plot(param_values, final_Rcb, 's-', 
                   linewidth=2, markersize=8, label='Bulk')
    axes[0, 1].plot(param_values, final_Rcf, '^-', 
                   linewidth=2, markersize=8, label='Boundary')
    axes[0, 1].set_xlabel(param_label)
    axes[0, 1].set_ylabel('Final Bubble Radius (nm)')
    axes[0, 1].set_title('Bubble Size')
    axes[0, 1].legend()
    axes[0, 1].grid(True, alpha=0.3)
    
    # Plot 3: Bubble concentration vs parameter
    axes[1, 0].semilogy(param_values, final_Ccb, 's-', 
                        linewidth=2, markersize=8, label='Bulk')
    axes[1, 0].semilogy(param_values, final_Ccf, '^-', 
                        linewidth=2, markersize=8, label='Boundary')
    axes[1, 0].set_xlabel(param_label)
    axes[1, 0].set_ylabel('Final Bubble Concentration (m⁻³)')
    axes[1, 0].set_title('Bubble Number Density')
    axes[1, 0].legend()
    axes[1, 0].grid(True, alpha=0.3)
    
    # Plot 4: Swelling evolution comparison
    for i, r in enumerate(results[::max(1, len(results)//5)]):
        label = f'{r[param_name]:.2e}'
        axes[1, 1].plot(r['time_days'], r['swelling'], 
                       linewidth=2, label=label)
    axes[1, 1].set_xlabel('Time (days)')
    axes[1, 1].set_ylabel('Swelling (%)')
    axes[1, 1].set_title('Swelling Evolution')
    axes[1, 1].legend(title=param_label, fontsize=8)
    axes[1, 1].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()
    
    # Print summary
    print(f"\nSummary for {param_name}:")
    print(f"  Range: {min(param_values):.2e} - {max(param_values):.2e}")
    print(f"  Swelling range: {min(final_swelling):.4f}% - {max(final_swelling):.4f}%")
    print(f"  Max swelling at {param_name}={param_values[np.argmax(final_swelling)]:.2e}")


print("Helper functions defined successfully!")

## 2. Temperature Sweep (Detailed Study)

Temperature is one of the most important parameters affecting gas swelling. Let's explore its effects in detail.

In [ ]:
# Define temperature range
temperatures = np.linspace(600, 1000, 9)  # 600-1000 K in 9 steps

# Run temperature sweep
temp_results = run_parameter_sweep(
    param_name='temperature',
    param_values=temperatures,
    sim_time_days=100
)

In [ ]:
# Visualize temperature sweep results
plot_sweep_results(temp_results, 'temperature', 'Temperature (K)')

### Physical Interpretation of Temperature Effects

The temperature dependence shows several key features:

1. **Peak swelling** at intermediate temperatures (~750-800 K)
   - Low T: Diffusion limits gas transport to bubbles
   - Intermediate T: Optimal balance for bubble growth
   - High T: Thermal vacancy emission reduces bubble stability

2. **Bubble size increases monotonically with T**
   - Faster diffusion enables larger bubbles
   - Gas pressure increases with T

3. **Bubble concentration may decrease with T**
   - Nucleation vs growth competition
   - At high T, growth dominates over nucleation

## 3. Fission Rate Sweep

The fission rate controls how fast gas atoms are produced. Let's investigate its effect on swelling.

In [ ]:
# Define fission rate range (typical reactor range)
fission_rates = np.array([1e19, 2e19, 4e19, 6e19, 8e19, 1e20])  # fissions/(m³·s)

# Run fission rate sweep at fixed temperature (near peak swelling)
base_params = create_default_parameters()
base_params['temperature'] = 750  # K (near peak swelling)

fr_results = run_parameter_sweep(
    param_name='fission_rate',
    param_values=fission_rates,
    base_params=base_params,
    sim_time_days=100
)

In [ ]:
# Visualize fission rate sweep results
plot_sweep_results(fr_results, 'fission_rate', 'Fission Rate (fissions/m³/s)')

### Key Observations: Fission Rate Effects

1. **Nearly linear relationship** between fission rate and swelling
   - More gas production → more swelling
   - However, some saturation may occur at very high rates

2. **Bubble size relatively insensitive** to fission rate
   - Bubble growth is transport-limited, not production-limited
   - Similar bubble sizes across the range

3. **Bubble concentration increases** with fission rate
   - More gas atoms → more nucleation sites
   - Higher number density of bubbles

## 4. Dislocation Density Sweep

Dislocations act as sinks for point defects. Let's explore how defect sink strength affects swelling.

In [ ]:
# Define dislocation density range
dislocation_densities = np.array([1e13, 3e13, 5e13, 7e13, 1e14, 3e14, 5e14])  # m⁻²

# Run dislocation density sweep
base_params = create_default_parameters()
base_params['temperature'] = 750  # K

dd_results = run_parameter_sweep(
    param_name='dislocation_density',
    param_values=dislocation_densities,
    base_params=base_params,
    sim_time_days=100
)

In [ ]:
# Visualize dislocation density results
plot_sweep_results(dd_results, 'dislocation_density', 'Dislocation Density (m⁻²)')

### Physical Interpretation: Dislocation Density Effects

Dislocations affect swelling through several mechanisms:

1. **Vacancy absorption**: More dislocations → more vacancy annihilation
   - This can **reduce swelling** at high dislocation densities
   - Vacancies are removed before they can reach bubbles

2. **Bias effect**: Dislocations have preference for interstitials
   - Creates vacancy supersaturation
   - Can **enhance swelling** at moderate densities

3. **Non-monotonic behavior**: Often see peak swelling at intermediate densities
   - Low density: Insufficient bias to create vacancy supersaturation
   - High density: Too much vacancy annihilation at dislocations
   - Intermediate: Optimal balance

## 5. Surface Energy Sweep

Surface energy affects the energy cost of creating bubble surfaces, influencing bubble size and nucleation.

In [ ]:
# Define surface energy range
surface_energies = np.array([0.3, 0.4, 0.5, 0.6, 0.8, 1.0])  # J/m²

# Run surface energy sweep
base_params = create_default_parameters()
base_params['temperature'] = 750  # K

se_results = run_parameter_sweep(
    param_name='surface_energy',
    param_values=surface_energies,
    base_params=base_params,
    sim_time_days=100
)

In [ ]:
# Visualize surface energy results
plot_sweep_results(se_results, 'surface_energy', 'Surface Energy (J/m²)')

### Surface Energy Effects

1. **Higher surface energy → smaller bubbles**
   - Energy penalty for surface area favors compact bubbles
   - Bubbles minimize surface to reduce energy

2. **Higher surface energy → may reduce swelling**
   - Smaller bubbles for same amount of gas
   - R³ dependence means size dominates over number

3. **Nucleation effects**:
   - Higher surface energy increases critical radius for nucleation
   - Fewer bubbles nucleate, but those that do grow differently

## 6. Nucleation Factor Sweep (Fnb - Bulk)

The nucleation factor controls the rate of new bubble formation. Let's explore bulk bubble nucleation.

In [ ]:
# Define nucleation factor range
nucleation_factors = np.array([1e-7, 1e-6, 1e-5, 5e-5, 1e-4, 5e-4])

# Run nucleation factor sweep
base_params = create_default_parameters()
base_params['temperature'] = 750  # K

fnb_results = run_parameter_sweep(
    param_name='Fnb',
    param_values=nucleation_factors,
    base_params=base_params,
    sim_time_days=100
)

In [ ]:
# Visualize nucleation factor results
plot_sweep_results(fnb_results, 'Fnb', 'Bulk Nucleation Factor (Fnb)')

### Nucleation Factor Effects

The nucleation factor creates an interesting **trade-off**:

1. **Low nucleation (Fnb small)**:
   - Fewer bubbles nucleate
   - Each bubble gets more gas → grows larger
   - Swelling ∝ R³ × Cc: Large R but small Cc

2. **High nucleation (Fnb large)**:
   - Many bubbles nucleate
   - Gas distributed among many bubbles → smaller size
   - Swelling ∝ R³ × Cc: Small R but large Cc

3. **Optimal nucleation**: Maximum swelling at intermediate values
   - Balance between bubble size and number
   - Product R³ × Cc is maximized

## 7. Comparison: Bulk vs Boundary Nucleation

Let's compare how bulk and boundary nucleation factors (Fnb and Fnf) affect swelling differently.

In [ ]:
# Compare Fnb (bulk) vs Fnf (boundary)
nucleation_range = np.array([1e-6, 1e-5, 1e-4, 1e-3])

# Sweep Fnb (bulk nucleation)
base_params = create_default_parameters()
base_params['temperature'] = 750
base_params['Fnf'] = 1e-5  # Fixed boundary nucleation

fnb_compare = run_parameter_sweep(
    param_name='Fnb',
    param_values=nucleation_range,
    base_params=base_params,
    sim_time_days=100
)

# Sweep Fnf (boundary nucleation)
base_params = create_default_parameters()
base_params['temperature'] = 750
base_params['Fnb'] = 1e-5  # Fixed bulk nucleation

fnf_compare = run_parameter_sweep(
    param_name='Fnf',
    param_values=nucleation_range,
    base_params=base_params,
    sim_time_days=100
)

In [ ]:
# Compare bulk vs boundary nucleation effects
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Extract swelling data
fnb_swellings = [r['swelling'][-1] for r in fnb_compare]
fnf_swellings = [r['swelling'][-1] for r in fnf_compare]

# Plot 1: Bulk nucleation effect
axes[0].semilogx(nucleation_range, fnb_swellings, 'o-', 
                linewidth=2, markersize=10, color='blue', label='Bulk')
axes[0].set_xlabel('Bulk Nucleation Factor (Fnb)')
axes[0].set_ylabel('Final Swelling (%)')
axes[0].set_title('Effect of Bulk Nucleation')
axes[0].grid(True, alpha=0.3)

# Plot 2: Boundary nucleation effect
axes[1].semilogx(nucleation_range, fnf_swellings, 's-', 
                linewidth=2, markersize=10, color='red', label='Boundary')
axes[1].set_xlabel('Boundary Nucleation Factor (Fnf)')
axes[1].set_ylabel('Final Swelling (%)')
axes[1].set_title('Effect of Boundary Nucleation')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("\nComparison Summary:")
print(f"  Bulk nucleation range: {min(fnb_swellings):.4f}% - {max(fnb_swellings):.4f}%")
print(f"  Boundary nucleation range: {min(fnf_swellings):.4f}% - {max(fnf_swellings):.4f}%")

### Bulk vs Boundary Nucleation

Key observations:

1. **Boundary nucleation typically has stronger effect** on swelling
   - Grain boundaries are preferential nucleation sites
   - Faster gas transport to boundaries

2. **Different swelling mechanisms**:
   - Bulk: Bubbles distributed throughout grains
   - Boundary: Bubbles concentrated at grain boundaries

3. **Design implication**: Controlling grain size can affect swelling
   - Smaller grains → more boundary area → more boundary swelling
   - Larger grains → more bulk swelling

## 8. Gas Production Rate Sweep

The gas production rate determines how many gas atoms are produced per fission event.

In [ ]:
# Define gas production rate range
# Typical range: 0.25 - 0.30 gas atoms per fission for U-Pu-Zr
gas_rates = np.array([0.20, 0.25, 0.30, 0.35, 0.40])

# Run gas production rate sweep
base_params = create_default_parameters()
base_params['temperature'] = 750  # K

gas_results = run_parameter_sweep(
    param_name='gas_production_rate',
    param_values=gas_rates,
    base_params=base_params,
    sim_time_days=100
)

In [ ]:
# Visualize gas production rate results
plot_sweep_results(gas_results, 'gas_production_rate', 'Gas Production Rate (atoms/fission)')

## 9. Sensitivity Analysis: Parameter Ranking

Now let's compare the relative sensitivity of swelling to different parameters.

In [ ]:
# Calculate normalized sensitivity for each parameter
def calculate_sensitivity(results, param_name):
    """
    Calculate normalized sensitivity: dS/dP * (P/S)
    
    Where S is swelling and P is parameter value
    """
    param_values = np.array([r[param_name] for r in results])
    swellings = np.array([r['swelling'][-1] for r in results])
    
    # Normalize to range [0, 1] for comparison
    param_norm = (param_values - param_values.min()) / (param_values.max() - param_values.min())
    swelling_norm = (swellings - swellings.min()) / (swellings.max() - swellings.min())
    
    # Sensitivity = change in swelling / change in parameter
    if len(param_norm) > 1:
        sensitivity = np.abs(np.polyfit(param_norm, swelling_norm, 1)[0])
    else:
        sensitivity = 0
    
    # Return variance in swelling as sensitivity metric
    return swellings.max() - swellings.min()


# Collect sensitivities for all parameters
sensitivities = {
    'Temperature': calculate_sensitivity(temp_results, 'temperature'),
    'Fission Rate': calculate_sensitivity(fr_results, 'fission_rate'),
    'Dislocation Density': calculate_sensitivity(dd_results, 'dislocation_density'),
    'Surface Energy': calculate_sensitivity(se_results, 'surface_energy'),
    'Nucleation (Fnb)': calculate_sensitivity(fnb_results, 'Fnb'),
    'Gas Production Rate': calculate_sensitivity(gas_results, 'gas_production_rate'),
}

# Plot sensitivity ranking
fig, ax = plt.subplots(figsize=(10, 6))

# Sort by sensitivity
sorted_items = sorted(sensitivities.items(), key=lambda x: x[1], reverse=True)
params = [item[0] for item in sorted_items]
values = [item[1] for item in sorted_items]

# Create horizontal bar plot
colors = plt.cm.viridis(np.linspace(0, 1, len(params)))
bars = ax.barh(params, values, color=colors)

# Customize plot
ax.set_xlabel('Swelling Variation (%)', fontsize=12, fontweight='bold')
ax.set_title('Parameter Sensitivity Ranking', fontsize=14, fontweight='bold')
ax.grid(True, axis='x', alpha=0.3)

# Add value labels on bars
for i, (bar, val) in enumerate(zip(bars, values)):
    ax.text(val + 0.01, bar.get_y() + bar.get_height()/2, 
            f'{val:.3f}%', va='center', fontsize=10)

plt.tight_layout()
plt.show()

print("\nParameter Sensitivity Ranking:")
print("=" * 50)
for i, (param, sens) in enumerate(sorted_items, 1):
    print(f"{i}. {param:20s}: ΔSwelling = {sens:.4f}%")

### Sensitivity Analysis Insights

The sensitivity ranking reveals:

1. **Most sensitive parameters**: These require accurate measurement
   - Small uncertainties → large prediction uncertainties
   - Priority for experimental validation

2. **Least sensitive parameters**: Can be approximated
   - Small errors have minimal impact
   - Can use literature values with confidence

3. **Implications for model validation**:
   - Focus calibration efforts on sensitive parameters
   - Uncertainty quantification should prioritize sensitive parameters

## 10. Two-Parameter Sweep: Temperature vs Fission Rate

Let's explore how two parameters interact by creating a 2D parameter map.

In [ ]:
# Define 2D parameter grid
temperatures_2d = np.linspace(650, 900, 6)
fission_rates_2d = np.array([1e19, 3e19, 5e19, 7e19, 1e20])

# Run 2D sweep
print("Running 2D parameter sweep (Temperature × Fission Rate)...")
print(f"Grid size: {len(temperatures_2d)} × {len(fission_rates_2d)} = {len(temperatures_2d) * len(fission_rates_2d)} simulations")
print("\nThis may take a few minutes...\n")

swelling_map = np.zeros((len(temperatures_2d), len(fission_rates_2d)))

for i, temp in enumerate(temperatures_2d):
    for j, fr in enumerate(fission_rates_2d):
        # Create parameters
        params = create_default_parameters()
        params['temperature'] = temp
        params['fission_rate'] = fr
        
        # Run simulation
        result = run_single_simulation(params, sim_time_days=100)
        swelling_map[i, j] = result['swelling'][-1]
        
        print(f"  [{i*len(fission_rates_2d)+j+1}/{len(temperatures_2d)*len(fission_rates_2d)}] "
              f"T={temp:.0f}K, FR={fr:.1e}: swelling={swelling_map[i,j]:.4f}%")

print("\n2D sweep completed!")

In [ ]:
# Visualize 2D parameter map
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Plot 1: Heatmap
im = axes[0].imshow(swelling_map, aspect='auto', origin='lower', cmap='viridis')
axes[0].set_xticks(range(len(fission_rates_2d)))
axes[0].set_yticks(range(len(temperatures_2d)))
axes[0].set_xticklabels([f'{fr:.1e}' for fr in fission_rates_2d], rotation=45)
axes[0].set_yticklabels([f'{T:.0f}' for T in temperatures_2d])
axes[0].set_xlabel('Fission Rate (fissions/m³/s)')
axes[0].set_ylabel('Temperature (K)')
axes[0].set_title('Swelling (%) Heatmap')
cbar = plt.colorbar(im, ax=axes[0])
cbar.set_label('Swelling (%)')

# Plot 2: Contour plot
X, Y = np.meshgrid(fission_rates_2d/1e19, temperatures_2d)
contour = axes[1].contour(X, Y, swelling_map, levels=10, colors='black', linewidths=0.5)
contourf = axes[1].contourf(X, Y, swelling_map, levels=20, cmap='viridis')
axes[1].set_xlabel('Fission Rate (×10¹⁹ fissions/m³/s)')
axes[1].set_ylabel('Temperature (K)')
axes[1].set_title('Swelling Contours')
plt.colorbar(contourf, ax=axes[1], label='Swelling (%)')

plt.tight_layout()
plt.show()

# Find optimal conditions
max_idx = np.unravel_index(np.argmax(swelling_map), swelling_map.shape)
print(f"\nMaximum swelling: {swelling_map[max_idx]:.4f}%")
print(f"  at T = {temperatures_2d[max_idx[0]]:.0f} K, FR = {fission_rates_2d[max_idx[1]]:.1e}")

### Two-Parameter Interaction Insights

The 2D map reveals parameter interactions:

1. **Temperature-fission rate coupling**:
   - Peak swelling temperature shifts with fission rate
   - Higher fission rates may shift peak to lower temperatures

2. **Operating window identification**:
   - Contour plots show regions of acceptable swelling
   - Helps define safe operating envelopes

3. **Design optimization**:
   - Maps can guide alloy design and operating conditions
   - Identify trade-offs between different objectives

## 11. Time Evolution: Parameter Effects Over Time

Let's examine how parameter effects evolve with irradiation time.

In [ ]:
# Compare swelling evolution for different temperatures at multiple times
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Plot 1: Swelling vs time at different temperatures
colors = plt.cm.plasma(np.linspace(0, 1, len(temp_results)))
for i, result in enumerate(temp_results):
    axes[0].plot(result['time_days'], result['swelling'],
                 linewidth=2, color=colors[i], 
                 label=f"{result['temperature']:.0f} K")
axes[0].set_xlabel('Time (days)')
axes[0].set_ylabel('Swelling (%)')
axes[0].set_title('Swelling Evolution at Different Temperatures')
axes[0].legend(ncol=2, fontsize=8)
axes[0].grid(True, alpha=0.3)

# Plot 2: Swelling rate vs time
for i, result in enumerate(temp_results[::2]):  # Every other temp
    # Calculate swelling rate (derivative)
    swelling_rate = np.gradient(result['swelling'], result['time_days'])
    axes[1].semilogy(result['time_days'][1:], swelling_rate[1:],
                     linewidth=2, color=colors[i*2],
                     label=f"{result['temperature']:.0f} K")
axes[1].set_xlabel('Time (days)')
axes[1].set_ylabel('Swelling Rate (%/day)')
axes[1].set_title('Swelling Rate Evolution')
axes[1].legend(fontsize=9)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("\nTime Evolution Observations:")
print("1. Early time: Nucleation-dominated, low swelling rates")
print("2. Intermediate: Growth-dominated, peak swelling rates")
print("3. Late time: Saturation or gas release, changing rates")

## 12. Summary and Guidelines for Parameter Studies

### What We Learned

1. **Temperature**: Most critical parameter with non-monotonic effect
   - Peak swelling at intermediate T (~750-800 K)
   - Requires careful optimization for reactor design

2. **Fission Rate**: Nearly linear effect on swelling
   - Higher gas production → more swelling
   - Relatively straightforward to model

3. **Dislocation Density**: Complex behavior through bias effects
   - Can increase or decrease swelling depending on range
   - Important for microstructure design

4. **Nucleation Factors**: Size vs number trade-off
   - Optimal nucleation for maximum swelling
   - Important for understanding bubble size distributions

5. **Surface Energy**: Affects bubble size and stability
   - Material property that can be modified
   - Important for alloy design

### Best Practices for Parameter Sweeps

1. **Start with sensitivity analysis**:
   - Identify important parameters first
   - Focus computational resources where they matter

2. **Use appropriate parameter ranges**:
   - Based on physical constraints
   - Literature values for similar materials
   - Experimental data when available

3. **Consider parameter interactions**:
   - 2D sweeps reveal coupling effects
   - Optimal conditions may depend on multiple parameters

4. **Think about time evolution**:
   - Early vs late time behavior can differ
   - Incubation periods, transient effects

5. **Validate with experiments**:
   - Compare predictions with measurements
   - Use sweeps to guide experimental design

### Next Steps

1. **Explore other notebooks**:
   - `03_Gas_Distribution_Analysis.ipynb`: Detailed gas tracking
   - `04_Experimental_Data_Comparison.ipynb`: Validation studies
   - `05_Custom_Material_Composition.ipynb`: Composition effects

2. **Try your own sweeps**:
   - Combine parameters in new ways
   - Explore extreme conditions
   - Model specific reactor scenarios

3. **Advanced analysis**:
   - Uncertainty quantification
   - Monte Carlo sampling
   - Response surface methodology

---

**🎉 Congratulations!** You've completed the parameter sweep studies tutorial.

You now have the skills to:
- Design systematic parameter studies
- Identify sensitive parameters
- Create sensitivity rankings
- Explore parameter interactions
- Optimize for specific objectives

Happy researching! 🚀